In [1]:
import os
import shutil
import re
import requests
import json
import nest_asyncio
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
from tika import parser
import pandas as pd

/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [2]:
from rag.lib.metadata import build_metadata
from rag.lib.config import LEXES_API_URL
from rag.lib.metadata import join_language

In [3]:
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

In [4]:
nest_asyncio.apply()

response = requests.get(LEXES_API_URL)
if response.status_code != 200:
    # TODO : a gerer dans les logs
    raise Exception(f"Unexpected status code while fetching : {response.status_code}")

data = build_metadata(response)

No SPARQL results for https://fedlex.data.admin.ch/redirectInfo?url=eli%2Fcc%2F1993%2F210_210_210%2Ffr&original=https:%2F%2Fwww.admin.ch%2Fch%2Ff%2Frs%2Fc414_110.html


In [5]:
# load all documents

def write_txt(filename, dir, content):
    with open(os.path.join(dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

def download_file(url, dir, filename):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147.0) Gecko/20100101 Firefox/147.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(os.path.join(dir, filename), "wb") as f:
            f.write(response.content)
    else:
        print(f"Error: the content for {url} can not be fetched (status: {response.status_code})")

shutil.rmtree(Path(DATA_DIR), ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

for content, metadata in data.items():
    doc_id = metadata.get("doc_id")
    content_format = metadata.get("content_format")
    if content_format == "txt":
        filename = f"{doc_id}.txt"
        write_txt(filename, DATA_DIR, content)
    elif content_format == "docx":
        filename = f"{doc_id}.docx"
        download_file(content, DATA_DIR, filename)
    elif content_format == "pdf":
        filename = f"{doc_id}.pdf"
        download_file(content, DATA_DIR, filename)
    else:
        print(f"File format not yet supported: '{content}'")

In [6]:
# filter pdf to index based on heuristic (np_pages + nb_occ_article) and hard coded values

ARTICLE_PATTERN = re.compile(r"\b(?:Article\s+\d+|Art\.\s*\d+)\b")

stats_per_doc = []

for file in tqdm(DATA_DIR.glob("*.pdf")):
    parsed_file = parser.from_file(str(file))
    metadata = parsed_file.get("metadata", {})
    nb_pages = int(metadata.get("xmpTPg:NPages"))
    extracted_text = parsed_file['content']
    nb_occ_article = sum(1 for _ in ARTICLE_PATTERN.finditer(extracted_text))
    stats_per_doc.append({
        "doc_id": file.stem,
        "nb_pages": nb_pages,
        "nb_occ_article": nb_occ_article
    })

stats = pd.DataFrame(stats_per_doc)
pdfs_not_to_index = stats.loc[(stats["nb_pages"] > 100) | ((stats["nb_pages"] > 50) & (stats["nb_occ_article"] == 0))]["doc_id"].values
print(pdfs_not_to_index)

0it [00:00, ?it/s]

<ArrowStringArray>
[]
Length: 0, dtype: str


In [7]:
# add key 'is_indexed' in metadata dict

for content, metadata_dict in data.items():
    metadata = metadata_dict
    doc_id = metadata["doc_id"]
    if doc_id in pdfs_not_to_index:
        metadata["is_indexed"] = False
    else:
        metadata["is_indexed"] = True
    data[content] = metadata

In [8]:
# load actual state of metadata in json file

os.makedirs(STATS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata.json")

with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)